In [3]:
import cv2
import os
import numpy as np

In [5]:

# ==========================================
#      設定エリア (ここを変更して使ってください)
# ==========================================

# 1. 加工したい画像のパス
INPUT_IMAGE_PATH = "images/test_pyramid2.jpeg" 

# 2. リサイズ設定 (長辺の最大ピクセル数)
# ※SAMの処理速度と精度のため、1000〜1500推奨
MAX_SIZE = 1500

# 3. コントラスト強調の強さ (CLAHE)
# 2.0 〜 4.0 くらいが目安。高いほど明るくクッキリしますが、ノイズも増えます。
CONTRAST_STRENGTH = 3.0

# 4. シャープ化を行うかどうか
# 輪郭がぼやけている場合は True
APPLY_SHARPEN = True

# ==========================================

def process_image():
    print(f"--- 画像加工プロセス開始: {INPUT_IMAGE_PATH} ---")

    # 1. 画像読み込み
    if not os.path.exists(INPUT_IMAGE_PATH):
        print(f"エラー: ファイルが見つかりません -> {INPUT_IMAGE_PATH}")
        return

    img = cv2.imread(INPUT_IMAGE_PATH)
    if img is None:
        print("エラー: 画像として読み込めませんでした。")
        return

    h, w = img.shape[:2]
    print(f"元画像サイズ: {w} x {h}")

    # 2. リサイズ処理
    if max(h, w) > MAX_SIZE:
        scale = MAX_SIZE / max(h, w)
        new_w, new_h = int(w * scale), int(h * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
        print(f"リサイズ実行: {new_w} x {new_h} (scale: {scale:.2f})")
    else:
        print("リサイズ不要: 指定サイズ以下です。")

    # 3. コントラスト強調 (CLAHE)
    # 明度情報(L)だけを取り出して強調し、色味(AB)は変えない手法
    try:
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        
        clahe = cv2.createCLAHE(clipLimit=CONTRAST_STRENGTH, tileGridSize=(8, 8))
        l_enhanced = clahe.apply(l)
        
        lab_enhanced = cv2.merge((l_enhanced, a, b))
        img = cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2BGR)
        print(f"コントラスト強調 (CLAHE): 強度 {CONTRAST_STRENGTH}")
    except Exception as e:
        print(f"警告: コントラスト強調に失敗しました ({e})")

    # 4. シャープ化 (Unsharp Masking風フィルタ)
    if APPLY_SHARPEN:
        # 中心を強調し、周囲を引くカーネル
        kernel = np.array([[0, -1, 0],
                           [-1, 5, -1],
                           [0, -1, 0]])
        img = cv2.filter2D(img, -1, kernel)
        print("シャープ化フィルタ: 適用完了")

    # 5. 保存
    root, ext = os.path.splitext(INPUT_IMAGE_PATH)
    output_path = f"{root}_processed{ext}"
    
    cv2.imwrite(output_path, img)
    print("-" * 30)
    print(f"加工完了！保存先: {output_path}")
    print("このパスをSAM解析のコードに入力してください。")

if __name__ == "__main__":
    process_image()

--- 画像加工プロセス開始: images/test_pyramid2.jpeg ---
元画像サイズ: 800 x 450
リサイズ不要: 指定サイズ以下です。
コントラスト強調 (CLAHE): 強度 3.0
シャープ化フィルタ: 適用完了
------------------------------
加工完了！保存先: images/test_pyramid2_processed.jpeg
このパスをSAM解析のコードに入力してください。
